In [3]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

RAW_DIR = Path("..") / "data" / "raw"
JSONL_DIR = RAW_DIR / "jsonl"
CSV_DIR = RAW_DIR / "csv"

print("RAW exists:", RAW_DIR.exists())
print("JSONL:", sorted(p.name for p in JSONL_DIR.glob("*")))
print("CSV:  ", sorted(p.name for p in CSV_DIR.glob("*")))

sample_file = sorted(JSONL_DIR.glob("*.jsonl"))[0]
print("file:", sample_file)

with open(sample_file, "r", encoding="utf-8") as f:
    first_line = f.readline()

record = json.loads(first_line)
print(json.dumps(record, ensure_ascii=False, indent=2)[:2000])

RAW exists: True
JSONL: ['hiddentest_dialogue.jsonl', 'hiddentest_topic.jsonl', 'test.jsonl', 'train.jsonl', 'validation.jsonl']
CSV:   ['hiddentest_dialogue.csv', 'hiddentest_topic.csv', 'test.csv', 'train.csv', 'validation.csv']
file: ..\data\raw\jsonl\hiddentest_dialogue.jsonl
{
  "fname": "hiddentest_0",
  "dialogue": "#Person1#: I have an issue.\n#Person2#: What is your problem?\n#Person1#: There were charges on my debit card that I never made.\n#Person2#: Do you have a statement for your debit card?\n#Person1#: I do.\n#Person2#: Which charges are you talking about?\n#Person1#: It's the last four charges.\n#Person2#: We're going to have to do an investigation.\n#Person1#: Is that going to take long?\n#Person2#: I'm not sure, but in the meantime we will freeze these charges.\n#Person1#: That's wonderful. Thank you."
}


In [4]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

splits = {}
for split in ("train", "val", "test"):
    # файл может называться train.jsonl или dialogsum.train.jsonl — покрываем оба
    candidates = list(JSONL_DIR.glob(f"*{split}*.jsonl"))
    if not candidates:
        print(f"[warn] no file for split={split}")
        continue
    df = load_jsonl(candidates[0])
    splits[split] = df
    print(f"{split:5s} <- {candidates[0].name}  shape={df.shape}")

train_df = splits["train"]
val_df = splits.get("val")
test_df = splits["test"]

train <- train.jsonl  shape=(12460, 4)
val   <- validation.jsonl  shape=(500, 4)
test  <- hiddentest_dialogue.jsonl  shape=(100, 2)


In [6]:
train_df

,fname,dialogue,summary,topic
0,train_0,"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You sho...","Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.",get a check-up
1,train_1,"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccin...",Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.,vaccines
2,train_2,"#Person1#: Excuse me, did you see a set of keys?\n#Person2#: What kind of keys?\n#Person1#: Five keys and a small foot ornament.\n#Person2#: What a shame! I didn't see them.\n#Person1#: Well, can ...",#Person1#'s looking for a set of keys and asks for #Person2#'s help to find them.,find keys
3,train_3,"#Person1#: Why didn't you tell me you had a girlfriend?\n#Person2#: Sorry, I thought you knew.\n#Person1#: But you should tell me you were in love with her.\n#Person2#: Didn't I?\n#Person1#: You k...",#Person1#'s angry because #Person2# didn't tell #Person1# that #Person2# had a girlfriend and would marry her.,have a girlfriend
4,train_4,"#Person1#: Watsup, ladies! Y'll looking'fine tonight. May I have this dance?\n#Person2#: He's cute! He looks like Tiger Woods! But, I can't dance. . .\n#Person1#: It's all good. I'll show you all ...",Malik invites Nikki to dance. Nikki agrees if Malik doesn't mind getting his feet stepped on.,dance
...,...,...,...,...
12455,train_12455,"#Person1#: Excuse me. You are Mr. Green from Manchester, aren't you?\n#Person2#: Yes, I'm glad you recognized me.\n#Person1#: A man with white hair and a beard, isn't that how you describe yoursel...",Tan Ling picks Mr. Green up who is easily recognized by white hair and a beard and will take him to the hotel. Tan reserves a big suite for him at the hotel.,pick up someone
12456,train_12456,"#Person1#: Mister Ewing said we should show up at the conference center at 4:00 o'clock, right?\n#Person2#: Yes, he especially asked us not to be late. Some of the people from our east york branch...",#Person1# and #Person2# plan to take the underground together to the conference center because Mr. Ewing asks them not to be late.,conference center
12457,train_12457,"#Person1#: How can I help you today?\n#Person2#: I would like to rent a car.\n#Person1#: Let's see what we can find. We have a large car, a mid-size car and a small car for you to choose from. Wha...",#Person2# rents a small car for 5 days with the help of #Person1#.,rent a car
12458,train_12458,"#Person1#: You look a bit unhappy today. What's up?\n#Person2#: Well, my mom lost her job yesterday.\n#Person1#: I'm sorry to hear that. Well, I heard that the registered urban unemployment rate r...",#Person2#'s mom lost her job. #Person2# hopes mom won't feel depressed. #Person1# suggests #Person2# searching for information on jobs on the internet.,job losing


In [7]:
train_df.info()
print("\ncolumns:", list(train_df.columns))
train_df.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 12460 entries, 0 to 12459
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   fname     12460 non-null  str  
 1   dialogue  12460 non-null  str  
 2   summary   12460 non-null  str  
 3   topic     12460 non-null  str  
dtypes: str(4)
memory usage: 389.5 KB

columns: ['fname', 'dialogue', 'summary', 'topic']


,fname,dialogue,summary,topic
0,train_0,"#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You sho...","Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.",get a check-up
1,train_1,"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccin...",Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.,vaccines
2,train_2,"#Person1#: Excuse me, did you see a set of keys?\n#Person2#: What kind of keys?\n#Person1#: Five keys and a small foot ornament.\n#Person2#: What a shame! I didn't see them.\n#Person1#: Well, can ...",#Person1#'s looking for a set of keys and asks for #Person2#'s help to find them.,find keys


In [19]:
test_df.info()
print("\ncolumns:", list(test_df.columns))
test_df.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   fname     100 non-null    str  
 1   dialogue  100 non-null    str  
dtypes: str(2)
memory usage: 1.7 KB

columns: ['fname', 'dialogue']


,fname,dialogue
0,hiddentest_0,#Person1#: I have an issue.\n#Person2#: What is your problem?\n#Person1#: There were charges on my debit card that I never made.\n#Person2#: Do you have a statement for your debit card?\n#Person1#...
1,hiddentest_1,"#Person1#: I was meaning to talk to you about something.\n#Person2#: What's going on?\n#Person1#: Were you having a party in your apartment last Friday?\n#Person2#: Yes, I had a little party.\n#Pe..."
2,hiddentest_2,"#Person1#: Hi, I am your new neighbor and was wondering if I could ask you a question or two.\n#Person2#: Hi! I would be happy to answer any questions you might have.\n#Person1#: Everyone here see..."


In [20]:
val_df.info()
print("\ncolumns:", list(val_df.columns))
val_df.head(3)

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   fname     500 non-null    str  
 1   dialogue  500 non-null    str  
 2   summary   500 non-null    str  
 3   topic     500 non-null    str  
dtypes: str(4)
memory usage: 15.8 KB

columns: ['fname', 'dialogue', 'summary', 'topic']


,fname,dialogue,summary,topic
0,dev_0,"#Person1#: Hello, how are you doing today?\n#Person2#: I ' Ve been having trouble breathing lately.\n#Person1#: Have you had any type of cold lately?\n#Person2#: No, I haven ' t had a cold. I just...",#Person2# has trouble breathing. The doctor asks #Person2# about it and will send #Person2# to a pulmonary specialist.,see a doctor
1,dev_1,#Person1#: Hey Jimmy. Let's go workout later today.\n#Person2#: Sure. What time do you want to go?\n#Person1#: How about at 3:30?\n#Person2#: That sounds good. Today we work on Legs and forearm.\n...,#Person1# invites Jimmy to go workout and persuades him into working out on arms and stomach.,do exercise
2,dev_2,#Person1#: I need to stop eating such unhealthy foods.\n#Person2#: I know what you mean. I've started eating better myself.\n#Person1#: What foods do you eat now?\n#Person2#: I tend to stick to fr...,"#Person1# plans to stop eating unhealthy foods, and #Person2# shares #Person2#'s healthy recipe with #Person1#.",healthy foods


In [9]:
train_df.topic.value_counts()

topic
shopping               174
job interview          161
daily casual talk      125
phone call              89
order food              79
                      ... 
look ill                 1
composition writing      1
conference center        1
job losing               1
baggage pack             1
Name: count, Length: 7434, dtype: int64

In [10]:
def add_len_stats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["dialogue_chars"] = df["dialogue"].str.len()
    df["summary_chars"] = df["summary"].str.len()
    df["dialogue_words"] = df["dialogue"].str.split().str.len()
    df["summary_words"] = df["summary"].str.split().str.len()
    df["compression"] = df["summary_words"] / df["dialogue_words"].replace(0, np.nan)
    return df

train_df = add_len_stats(train_df)
train_df[["dialogue_chars", "summary_chars", "dialogue_words", "summary_words", "compression"]].describe()

,dialogue_chars,summary_chars,dialogue_words,summary_words,compression
count,12460.000000,12460.000000,12460.000000,12460.000000,12460.000000
mean,738.281300,140.733547,130.986998,22.867657,0.185301
std,396.544752,65.133972,70.976634,10.705946,0.049095
min,190.000000,31.000000,40.000000,5.000000,0.051923
25%,485.000000,95.000000,85.000000,15.000000,0.155738
50%,653.000000,129.000000,116.000000,21.000000,0.181086
75%,916.000000,173.000000,163.000000,28.000000,0.205556
max,5178.000000,1039.000000,985.000000,153.000000,0.775510


In [11]:
print("Nulls per column:\n", train_df[["dialogue", "summary"]].isna().sum())
print("\nEmpty dialogues:", (train_df["dialogue"].str.strip() == "").sum())
print("Empty summaries:", (train_df["summary"].str.strip() == "").sum())

print("\nExact duplicate dialogues:", train_df["dialogue"].duplicated().sum())
print("Exact duplicate summaries:", train_df["summary"].duplicated().sum())
print("Exact duplicate pairs:",
      train_df[["dialogue", "summary"]].duplicated().sum())

# пересечение train/test по диалогам — важно для честной оценки
if test_df is not None:
    overlap = set(train_df["dialogue"]) & set(test_df["dialogue"])
    print("\nDialogue overlap train/test:", len(overlap))

Nulls per column:
 dialogue    0
summary     0
dtype: int64

Empty dialogues: 0
Empty summaries: 0

Exact duplicate dialogues: 2
Exact duplicate summaries: 24
Exact duplicate pairs: 0

Dialogue overlap train/test: 0


In [21]:
print("Nulls per column:\n", val_df[["dialogue", "summary"]].isna().sum())
print("\nEmpty dialogues:", (val_df["dialogue"].str.strip() == "").sum())
print("Empty summaries:", (val_df["summary"].str.strip() == "").sum())

print("\nExact duplicate dialogues:", val_df["dialogue"].duplicated().sum())
print("Exact duplicate summaries:", val_df["summary"].duplicated().sum())
print("Exact duplicate pairs:",
      val_df[["dialogue", "summary"]].duplicated().sum())

# пересечение train/test по диалогам — важно для честной оценки
if test_df is not None:
    overlap = set(val_df["dialogue"]) & set(val_df["dialogue"])
    print("\nDialogue overlap train/test:", len(overlap))

Nulls per column:
 dialogue    0
summary     0
dtype: int64

Empty dialogues: 0
Empty summaries: 0

Exact duplicate dialogues: 0
Exact duplicate summaries: 0
Exact duplicate pairs: 0

Dialogue overlap train/test: 500


In [15]:
train_df[train_df["dialogue"].duplicated(keep=False)]

,fname,dialogue,summary,topic,dialogue_chars,summary_chars,dialogue_words,summary_words,compression
1626,train_1626,"#Person1#: Any plans tonight? \n#Person2#: Not really, do you? \n#Person1#: Well, I am wondering if we took a hang-out for a drink or something. You know, I just came back from a really tough inte...",#Person1# invites #Person2# to have a drink because #Person1# had a tough interview. #Person1# tells that he did well in the paper exam but the questions he was asked were strange.,social communication,860,180,158,31,0.196203
7618,train_7618,"#Person1#: I'm searching for an old music box. \n#Person2#: You came to the right place. Any particular decade? \n#Person1#: If you had a box made in the '20s, that would be nice. \n#Person2#: We ...",#Person1# is searching for an old music box made in the '20s with dancing figures. #Person2# recommends two and #Person1# chooses one.,old music box,719,134,127,22,0.173228
9118,train_9118,"#Person1#: Any plans tonight? \n#Person2#: Not really, do you? \n#Person1#: Well, I am wondering if we took a hang-out for a drink or something. You know, I just came back from a really tough inte...",#Person1# invites #Person2# to take a hang-out after the tough interview. #Person1# thinks #Person1# did well in the paper exams but the interview wasn't as expected so #Person1# is worried about it.,after inteview,860,199,158,32,0.202532
11492,train_11492,"#Person1#: I'm searching for an old music box. \n#Person2#: You came to the right place. Any particular decade? \n#Person1#: If you had a box made in the '20s, that would be nice. \n#Person2#: We ...",#Person1# wants to buy an old music box with dancing figures and the shop assistant stops #Person1# from asking questions.,music box,719,122,127,20,0.157480


In [18]:
train_df[train_df["summary"].duplicated(keep=False)].sort_values("summary")

,fname,dialogue,summary,topic,dialogue_chars,summary_chars,dialogue_words,summary_words,compression
6189,train_6189,"#Person1#: How much does it cost for a bus pass?\n#Person2#: It'll be $ 65 for a monthly pass.\n#Person1#: Is there anything cheaper than that?\n#Person2#: If you're in school, you can get a stude...",#Person1# asks #Person2# some questions about the bus pass and then gets a student bus pass with #Person2#'s help.,bus pass,541,114,96,19,0.197917
6640,train_6640,"#Person1#: How much for a bus pass?\n#Person2#: Well, for a monthly pass, it'll cost you $ 65.\n#Person1#: Is there anything else that doesn't cost as much?\n#Person2#: If you're a student, you ca...",#Person1# asks #Person2# some questions about the bus pass and then gets a student bus pass with #Person2#'s help.,bus pass,583,114,104,19,0.182692
6312,train_6312,#Person1#: I would like to get some meat today.\n#Person2#: What kind do you need?\n#Person1#: I need about a pound of ground beef.\n#Person2#: The ground beef is $ 2. 48 a pound.\n#Person1#: That...,#Person1# buys a pound of ground beef and three pounds of chicken breasts from #Person2#.,shop for meat,558,89,100,15,0.150000
8989,train_8989,"#Person1#: I would like to purchase some meat.\n#Person2#: What kind of meat would you like to get today?\n#Person1#: First off, I'm going to need a pound of ground beef.\n#Person2#: A pound of gr...",#Person1# buys a pound of ground beef and three pounds of chicken breasts from #Person2#.,buy meat,547,89,96,15,0.156250
5146,train_5146,#Person1#: I met a girl in elevator this morning. She works in a company upstairs. She said she envied the casual way we dress for work in our company. She complained about the dress rule in her c...,#Person1# complains about the strict dressing code that the company upstairs has because the comfortable clothes lead to less productivity. #Person2# thinks professional attire makes a good impres...,dressing rule,947,201,170,28,0.164706
1237,train_1237,#Person1#: I met a girl in elevator this morning. She works in a company upstairs. She said she envied the casual way we dress for work. She complained about the dress rule in her company. It requ...,#Person1# complains about the strict dressing code that the company upstairs has because the comfortable clothes lead to less productivity. #Person2# thinks professional attire makes a good impres...,dress for work,857,201,151,28,0.185430
7862,train_7862,#Person1#: There is a tornado warning on. My mother just told me she heard it on the radio. \n#Person2#: What is a tornado warning? \n#Person1#: It means that a tornado has been seen somewhere in ...,#Person1# explains to #Person2# the details about tornados in terms of two stages which are a tornado warning and a tornado watch. #Person1# further explains the sound of tornados and the causes o...,discuss tornadoes,2101,371,365,60,0.164384
250,train_250,#Person1#: There is a tornado warning on. My mother just told me she heard it on the radio.\n#Person2#: What is a tornado warning?\n#Person1#: It means that a tornado has been seen somewhere in th...,#Person1# explains to #Person2# the details about tornados in terms of two stages which are a tornado warning and a tornado watch. #Person1# further explains the sound of tornados and the causes o...,a tornado warning,2078,371,365,60,0.164384
8822,train_8822,"#Person1#: Excuse me, sir, didn't you see the red light?\n#Person2#: Oh, I thought I could make a right turn on red there.\n#Person1#: No, sir. The sign says 'No Turn on Red'.\n#Person2#: Oh, I gu...",#Person1# gives #Person2# a ticket because he made a right turn on red where there is a sign saying 'No Turn on Red'.,give a ticket,533,117,96,23,0.239583
11236,train_11236,"#Person1#: Excuse me, sir. Didn't you see the red light?\n#Person2#: Oh, I thought I could make a right turn on red here.\n#Person1#: No, sir. The sign says no turn on red.\n#Person2#: Oh, I guess...",#Person1# gives #Person2# a ticket because he made a right turn on re

In [16]:
dup_sum_mask = train_df["summary"].duplicated(keep=False)
dup_sum = train_df[dup_sum_mask].sort_values("summary")

print("Строк с повторяющимся summary:", len(dup_sum))
print("Уникальных summary среди них:", dup_sum["summary"].nunique())

# распределение: сколько разных диалогов на одно summary
counts = dup_sum.groupby("summary").size()
print("\nСколько диалогов на одно summary:")
print(counts.value_counts().sort_index())

# средняя длина повторяющихся summary vs всех
print("\nСредняя длина summary (chars):")
print("повторяющиеся:", dup_sum["summary"].str.len().mean())
print("все:         ", train_df["summary"].str.len().mean())

Строк с повторяющимся summary: 48
Уникальных summary среди них: 24

Сколько диалогов на одно summary:
2    24
Name: count, dtype: int64

Средняя длина summary (chars):
повторяющиеся: 116.625
все:          140.7335473515249


# Вывод EDA
1. Пропусков в train, val, test нет  
2. Средняя длина диалога и саммари 131 и 28 соответвенно  
3. Есть одинаковые диалоги 2 штуки, но это нормально, так как саммари разное. Возможно разные разметчики сделали разное самари у одинаковых диалогов.  
4. Есть одинаковые саммари 24 шт, но диалоги у них разные. Это тоже нормально.  